# Blink Feature Extraction Tutorial

This tutorial demonstrates how to analyze a long continuous EAR/EOG recording and compute blink features every 30 seconds using the `pyear` package.

We use the sample `ear_eog.fif` file that accompanies the unit tests.

In [ ]:
from pathlib import Path

import mne
import pandas as pd
from tqdm import tqdm

from pyear.pipeline import extract_features

## 1. Load the raw recording

In [ ]:
fif_path = Path("../unitest/ear_eog.fif")
raw = mne.io.read_raw_fif(fif_path, preload=True)
print(f"Sampling rate: {raw.info["sfreq"]} Hz")

## 2. Slice the continuous signal into 30-second segments

In [ ]:
sfreq = raw.info["sfreq"]
epoch_len = 30.0
end_time = raw.times[-1]
n_epochs = int(end_time // epoch_len)
segments = []
for idx in tqdm(range(n_epochs), desc="Creating segments"):
    start = idx * epoch_len
    stop = start + epoch_len
    segment = raw.copy().crop(tmin=start, tmax=stop, include_tmax=False)
    shifted = mne.Annotations(segment.annotations.onset - start,
                              segment.annotations.duration,
                              segment.annotations.description)
    segment.set_annotations(shifted)
    segments.append(segment)

## 3. Convert annotations to blink dictionaries

In [ ]:
blinks = []
for idx, segment in enumerate(segments):
    signal = segment.get_data(picks="EAR-avg_ear")[0]
    ann = segment.annotations
    for onset, dur, desc in zip(ann.onset, ann.duration, ann.description):
        if desc != "blink":
            continue
        start_frame = int(onset * sfreq)
        end_frame = int((onset + dur) * sfreq)
        blinks.append({
            "refined_start_frame": start_frame,
            "refined_peak_frame": (start_frame + end_frame) // 2,
            "refined_end_frame": end_frame,
            "epoch_signal": signal,
            "epoch_index": idx,
        })

## 4. Compute features for each segment

In [ ]:
df = extract_features(blinks, sfreq, epoch_len, n_epochs, raw_segments=segments)
df.head()

The resulting DataFrame contains blink counts, kinematic metrics and other aggregated statistics for each 30-second epoch.